# pandas 3.0: Lo que cambió y por qué importa

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/14_pandas_transformaciones/code/04_pandas3.ipynb)

pandas 3.0 salió en enero 2025. Si ya sabes pandas 2.x, la mayoría del código sigue funcionando — pero hay cambios que pueden romper cosas silenciosamente o que ya lanzaban warnings y ahora lanzan errores.

Este notebook cubre los cambios que más importan, con código que puedes correr directamente.

**Estructura:**
0. Setup y verificación
1. Copy-on-Write (CoW) — el más importante
2. Strings con Arrow — nuevo dtype por default
3. `applymap` eliminado → `map`
4. `groupby` con `observed=True` por default
5. `DataFrame.append()` eliminado → `pd.concat()`
6. `inplace=` — estado de deprecación
7. Nullable dtypes más consistentes
8. Checklist de migración
9. Ejercicio

In [1]:
# Instalar pandas 3.x explícitamente
%pip install "pandas>=3.0.0" "numpy>=1.26.4" "pyarrow>=14.0.0"

  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
Note: you may need to restart the kernel to use updated packages.


## Sección 0: Setup y verificación de versión

In [2]:
import pandas as pd
import numpy as np

print(f"pandas version: {pd.__version__}")
print(f"numpy version:  {np.__version__}")

assert int(pd.__version__.split(".")[0]) >= 3, (
    "Este notebook requiere pandas 3.0+. "
    "Reinicia el kernel después de instalar."
)

pandas version: 3.0.1
numpy version:  1.26.4


---
## Sección 1: Copy-on-Write (CoW) — el cambio más importante

**¿Qué es CoW?** Antes, cuando hacías un subconjunto de un DataFrame, pandas a veces creaba una copia y a veces una vista — y no era predecible cuál. Eso significaba que modificar el subconjunto *podía o no* modificar el original, dependiendo de cómo lo hicieras.

**En pandas 2.x:** CoW era opt-in (`pd.options.mode.copy_on_write = True`). Si no lo activabas, seguías con el comportamiento impredecible más el `SettingWithCopyWarning`.

**En pandas 3.0:** CoW está siempre encendido, no se puede deshabilitar. La regla es simple y consistente: **cualquier subconjunto es su propio objeto independiente**. Modificar el subconjunto nunca modifica el original.

In [3]:
# Setup del DataFrame de ejemplo
df = pd.DataFrame({
    "nombre": ["Ana", "Carlos", "Maria"],
    "salario": [35000, 52000, 41000],
    "activo":  [True, True, False]
})

print("DataFrame original:")
print(df)

DataFrame original:
   nombre  salario  activo
0     Ana    35000    True
1  Carlos    52000    True
2   Maria    41000   False


In [4]:
# PATRON ANTIGUO — en pandas < 2.0 este código podia modificar df
# generaba SettingWithCopyWarning. En pandas 3.0: comportamiento claro.
subset = df[df["activo"]]
subset["salario"] = subset["salario"] * 1.1  # En pandas 3.0: modifica subset, NO df

print("df['salario'] despues de modificar subset:")
print(df["salario"].values)  # Sin cambios — CoW garantiza esto
print()
print("subset['salario'] (modificado):")
print(subset["salario"].values)

df['salario'] despues de modificar subset:
[35000 52000 41000]

subset['salario'] (modificado):
[38500. 57200.]


In [ ]:
# Para modificar df, usa .loc explícitamente
# Esto es el patrón correcto en pandas 3.0
df_copy = df.copy()

df_copy["salario"] = df_copy["salario"].astype(float)

df_copy.loc[df_copy["activo"], "salario"] = (
    df_copy.loc[df_copy["activo"], "salario"] * 1.1
)

print("df_copy con .loc (correcto):")
print(df_copy["salario"].values)
print()

print("df original sin cambios:")
print(df["salario"].values)


df_copy con .loc (correcto):
[38500. 57200. 41000.]

df original sin cambios:
[35000 52000 41000]


In [9]:
# CoW hace el method chaining eficiente
# No crea copias innecesarias en cada paso de la cadena
resultado = (
    df
    .assign(salario_anual=lambda d: d["salario"] * 12)
    .query("activo == True")
    .sort_values("salario_anual", ascending=False)
)

print(resultado)
print()
print("df original sigue igual — CoW garantiza el aislamiento:")
print(df)

   nombre  salario  activo  salario_anual
1  Carlos    52000    True         624000
0     Ana    35000    True         420000

df original sigue igual — CoW garantiza el aislamiento:
   nombre  salario  activo
0     Ana    35000    True
1  Carlos    52000    True
2   Maria    41000   False


**Por qué se hizo este cambio:** El comportamiento anterior era una fuente constante de bugs difíciles de encontrar. El código parecía funcionar en pruebas pequeñas pero fallaba con DataFrames más grandes porque la decisión de copia vs vista dependía de la implementación interna, no de lo que escribías. CoW hace el comportamiento 100% predecible.

**Impacto en performance:** En general CoW es más rápido para method chaining (el caso más común). En casos donde antes se beneficiaba de la vista (modificación in-place de grandes DataFrames), hay un overhead pequeño al primer write.

**💡 Prueba esto:** ¿Qué pasa si asignas `resultado["col_nueva"] = 0` después de la cadena de arriba? ¿Afecta a `df`?

---
## Sección 2: Strings con Arrow — nuevo dtype por default

**En pandas 2.x:** Las columnas de texto se almacenaban como `object` — básicamente un array de punteros Python a strings Python. Lento, usa mucha memoria.

**En pandas 3.0:** Las columnas de texto se almacenan como `StringDtype` por default. Internamente usa Arrow (columnar, más eficiente). El tipo que verás es `string[python]` o `string[pyarrow]` dependiendo de si tienes pyarrow instalado.

In [10]:
from io import StringIO

csv = """nombre,ciudad,edad
Ana,CDMX,25
Carlos,MTY,32
Maria,GDL,28"""

df_csv = pd.read_csv(StringIO(csv))

print("Tipos en pandas 3.0:")
print(df_csv.dtypes)
print()
print("En pandas 2.x, nombre y ciudad eran 'object'.")
print("En pandas 3.0, son StringDtype.")

Tipos en pandas 3.0:
nombre      str
ciudad      str
edad      int64
dtype: object

En pandas 2.x, nombre y ciudad eran 'object'.
En pandas 3.0, son StringDtype.


In [12]:
# Comparar uso de memoria: StringDtype vs object
s_string = df_csv["nombre"]                # StringDtype (pandas 3.0 default)
s_object = df_csv["nombre"].astype(object) # object (pandas 2.x style)

mem_string = s_string.memory_usage(deep=True)
mem_object = s_object.memory_usage(deep=True)

print(f"StringDtype: {mem_string} bytes")
print(f"object:      {mem_object} bytes")
print(f"Ratio:       {mem_object / mem_string:.2f}x más memoria con object")

StringDtype: 170 bytes
object:      293 bytes
Ratio:       1.72x más memoria con object


In [13]:
# StringDtype usa pd.NA, no np.nan
s = pd.Series(["Ana", None, "Carlos"], dtype="string")
print("Serie con NA:")
print(s)
print()
print("isna():")
print(s.isna())
print()

# pd.NA propaga a través de operaciones — comportamiento predecible
print(f"pd.NA + 'texto'  = {pd.NA + 'texto'}")   # <NA>
print(f"np.nan + 'texto' = ", end="")
try:
    print(np.nan + "texto")
except TypeError as e:
    print(f"TypeError: {e}")

Serie con NA:
0       Ana
1      <NA>
2    Carlos
dtype: string

isna():
0    False
1     True
2    False
dtype: bool

pd.NA + 'texto'  = <NA>
np.nan + 'texto' = TypeError: unsupported operand type(s) for +: 'float' and 'str'


**Por qué se hizo este cambio:** `object` nunca fue un buen tipo para strings. Era un contenedor genérico de Python que resultaba en acceso lento y uso de memoria innecesario. Arrow es una especificación columnar pensada para datos de texto.

**Impacto en código existente:** La mayoría del código funciona igual. Donde puede romper: código que checa `dtype == object` o que usa `np.nan` explícitamente para valores nulos en columnas de texto.

**💡 Si necesitas el comportamiento legacy:** `pd.read_csv(archivo, dtype_backend='numpy_nullable')` o incluso `dtype_backend='numpy'` para el comportamiento más antiguo.

---
## Sección 3: `applymap` eliminado → `map`

**En pandas 2.x:** `DataFrame.applymap(func)` aplicaba una función elemento por elemento a todo el DataFrame. En la 2.1 se deprecó y se renombró a `DataFrame.map()`.

**En pandas 3.0:** `applymap` ya no existe. Si tu código lo usa, lanza `AttributeError`.

In [14]:
df_numerico = pd.DataFrame({
    "precio":    [100.123, 200.456, 300.789],
    "descuento": [0.101,   0.202,   0.303],
})

# ESTO FALLA EN PANDAS 3.0:
# df_numerico.applymap(lambda x: round(x, 2))  # AttributeError!

# CORRECTO en pandas 3.0 — misma firma, mismo comportamiento:
df_redondeado = df_numerico.map(lambda x: round(x, 2))
print("Con df.map() (correcto):")
print(df_redondeado)

Con df.map() (correcto):
   precio  descuento
0  100.12        0.1
1  200.46        0.2
2  300.79        0.3


In [15]:
# Detectar y manejar código legacy que usaba applymap
# Útil si heredas código que no sabes si fue actualizado
try:
    df_numerico.applymap(lambda x: x)
except AttributeError as e:
    print(f"Error de compatibilidad detectado: {e}")
    print("Solución: reemplaza df.applymap(f) con df.map(f)")
    # Fix automático:
    resultado = df_numerico.map(lambda x: x)
    print("Fix aplicado correctamente.")

Error de compatibilidad detectado: 'DataFrame' object has no attribute 'applymap'
Solución: reemplaza df.applymap(f) con df.map(f)
Fix aplicado correctamente.


**La migración es trivial:** `applymap` → `map`, mismos argumentos, mismo resultado. El único esfuerzo es encontrar todas las ocurrencias en tu código. Usa `grep -r applymap .` en tu proyecto.

**Nota:** `Series.map()` existía antes y hace algo diferente (mapea valores usando un diccionario o función). `DataFrame.map()` es nuevo y aplica elemento por elemento. No los confundas.

---
## Sección 4: `groupby` con `observed=True` por default

Este es un **cambio silencioso** — no lanza error, solo cambia los resultados. Fácil de no notar.

**El contexto:** Cuando tienes una columna `Categorical` con categorías predefinidas, puedes tener categorías que no aparecen en los datos (ej: tienes categorías [A, B, C, D] pero tus datos solo tienen [A, B]).

- `observed=False` (comportamiento pandas 2.x): el groupby incluye todas las categorías, con NaN para las que no tienen datos
- `observed=True` (default pandas 3.0): el groupby solo muestra categorías que realmente tienen datos

In [18]:
# Setup: columna categórica con una categoría sin datos
genero = pd.Categorical(
    ["M", "M", "F", "F", "NB"],
    categories=["M", "F", "NB", "Prefiero no decir"]  # 4 categorías definidas
)
df_cat = pd.DataFrame({
    "genero":  genero,
    "salario": [50000, 55000, 48000, 52000, 47000]
})

print("Categorías definidas:", df_cat["genero"].cat.categories.tolist())
print("Categorías en los datos:", df_cat["genero"].unique().tolist())
print()
print(df_cat)

Categorías definidas: ['M', 'F', 'NB', 'Prefiero no decir']
Categorías en los datos: ['M', 'F', 'NB']

  genero  salario
0      M    50000
1      M    55000
2      F    48000
3      F    52000
4     NB    47000


In [19]:
# pandas 3.0 (observed=True por default):
# Solo muestra categorías con datos
resultado_default = df_cat.groupby("genero")["salario"].mean()

print("pandas 3.0 (observed=True, default):")
print(resultado_default)
print()
print("'Prefiero no decir' NO aparece — no hay filas con ese valor.")

pandas 3.0 (observed=True, default):
genero
M     52500.0
F     50000.0
NB    47000.0
Name: salario, dtype: float64

'Prefiero no decir' NO aparece — no hay filas con ese valor.


In [20]:
# Si necesitas TODAS las categorías (comportamiento de pandas 2.x):
resultado_todas = df_cat.groupby("genero", observed=False)["salario"].mean()

print("Con observed=False (explícito — comportamiento pandas 2.x):")
print(resultado_todas)
print()
print("'Prefiero no decir' aparece con NaN (0 observaciones).")

Con observed=False (explícito — comportamiento pandas 2.x):
genero
M                    52500.0
F                    50000.0
NB                   47000.0
Prefiero no decir        NaN
Name: salario, dtype: float64

'Prefiero no decir' aparece con NaN (0 observaciones).


**Por qué es peligroso este cambio:** Si en pandas 2.x tenías código que dependía de que las categorías sin datos aparecieran en el resultado (ej: un reporte que siempre muestra todas las regiones aunque algunas tengan 0 ventas), ese código ahora produce resultados incorrectos sin ningún error ni warning.

**Cómo detectarlo:** Busca en tu código cualquier `groupby` sobre columnas categóricas donde no se especifica `observed=`. Si el resultado tenía filas con NaN, probablemente dependías del comportamiento `observed=False`.

**💡 Prueba esto:** ¿En qué casos reales necesitas `observed=False`? Piensa en reportes donde necesitas mostrar que una categoría tuvo 0 ventas, o en tablas de contingencia donde todas las combinaciones deben aparecer.

---
## Sección 5: `DataFrame.append()` eliminado → `pd.concat()`

`DataFrame.append()` se deprecó en pandas 1.4 y se eliminó en pandas 2.0. Si llegas a pandas 3.0 con código que aún lo usa, lanzará `AttributeError`.

Lo incluimos aquí porque es muy común en código heredado y vale la pena tenerlo en el checklist.

In [21]:
df1 = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
df2 = pd.DataFrame({"a": [5, 6], "b": [7, 8]})

# ESTO FALLA EN PANDAS 2.0+ Y 3.0:
# df1.append(df2)  # AttributeError: 'DataFrame' object has no attribute 'append'

# CORRECTO:
df_combined = pd.concat([df1, df2], ignore_index=True)
print("Resultado de pd.concat:")
print(df_combined)

Resultado de pd.concat:
   a  b
0  1  3
1  2  4
2  5  7
3  6  8


In [22]:
# El anti-patrón más común con append: construir un DataFrame en un loop
# MUY LENTO — cuadrático en tiempo porque crea un DataFrame nuevo en cada iteración
#
# NO hagas esto (aunque encuentres este patrón en código legacy):
# df_acum = pd.DataFrame()
# for i in range(1000):
#     df_acum = df_acum.append({"x": i, "y": i**2}, ignore_index=True)  # O(n^2)

# CORRECTO: acumula en lista, concat al final (una sola vez)
filas = []
for i in range(5):
    filas.append({"x": i, "y": i**2})  # lista de dicts, no df.append()

df_final = pd.DataFrame(filas)  # O DataFrame(filas) o pd.concat de DataFrames
print("DataFrame construido con lista + pd.DataFrame:")
print(df_final)
print()
print("Este patrón es O(n) en lugar de O(n^2).")

DataFrame construido con lista + pd.DataFrame:
   x   y
0  0   0
1  1   1
2  2   4
3  3   9
4  4  16

Este patrón es O(n) en lugar de O(n^2).


---
## Sección 6: `inplace=` — estado de deprecación

**Situación actual:** `inplace=True` todavía existe en varios métodos de pandas 3.0, pero la dirección del proyecto es eliminarlo eventualmente. Ya desapareció de algunos métodos y en otros genera warnings.

**El problema con `inplace=`:** No es más rápido (pandas igual crea objetos internos), hace el código más difícil de leer en method chaining, y complica la implementación de CoW.

**La alternativa es siempre mejor:** reasignar la variable.

In [23]:
df_ejemplo = pd.DataFrame({
    "a": [3, 1, 2],
    "b": ["c", "a", "b"]
})

# PATRON A EVITAR (inplace):
# df_ejemplo.sort_values("a", inplace=True)
# df_ejemplo.reset_index(drop=True, inplace=True)

# PATRON CORRECTO (reasignar — siempre más claro y compatible):
df_ejemplo = df_ejemplo.sort_values("a").reset_index(drop=True)

print("DataFrame ordenado (sin inplace):")
print(df_ejemplo)

DataFrame ordenado (sin inplace):
   a  b
0  1  a
1  2  b
2  3  c


In [24]:
# Comparación de patrones para operaciones comunes
df_temp = pd.DataFrame({"x": [5, 3, 1, 4, 2], "y": ["e", "c", "a", "d", "b"]})

# Antiguo (inplace):
# df_temp.sort_values("x", inplace=True)
# df_temp.rename(columns={"x": "valor"}, inplace=True)
# df_temp.dropna(inplace=True)

# Nuevo (reasignar — también permite method chaining):
df_temp = (
    df_temp
    .sort_values("x")
    .rename(columns={"x": "valor"})
    .reset_index(drop=True)
)

print("Resultado con reasignación + chaining:")
print(df_temp)

Resultado con reasignación + chaining:
   valor  y
0      1  a
1      2  b
2      3  c
3      4  d
4      5  e


**Por qué se está eliminando `inplace=`:** Con CoW activado permanentemente, el concepto de modificar un objeto "en su lugar" no tiene mucho sentido — pandas de todas formas puede necesitar crear un objeto nuevo internamente. La reasignación es semánticamente más honesta sobre lo que está pasando.

**Regla práctica:** Si ves `inplace=True` en código nuevo, reemplázalo. Si ves `inplace=True` en código heredado que funciona, puedes dejarlo por ahora pero agrégalo al backlog de limpieza.

---
## Sección 7: Nullable dtypes — más consistentes en pandas 3.0

Los nullable dtypes (`Int64`, `Float64`, `boolean`, `string` con mayúscula/capitalización específica) ya existían desde pandas 1.x pero eran experimentales. En pandas 3.0 son estables y están bien integrados.

**La diferencia con los dtypes normales:** Los dtypes de NumPy (`int64`, `float64`) no pueden representar valores faltantes en enteros — tenían que upcastear a `float64` para meter `np.nan`. Los nullable dtypes usan `pd.NA` y mantienen el tipo original.

In [25]:
# Los 4 nullable dtypes más útiles
df_nullable = pd.DataFrame({
    "enteros_con_na": pd.array([1, 2, None, 4], dtype="Int64"),
    "floats_con_na":  pd.array([1.1, None, 3.3, 4.4], dtype="Float64"),
    "bool_con_na":    pd.array([True, None, False, True], dtype="boolean"),
    "texto_con_na":   pd.array(["a", "b", None, "d"], dtype="string"),
})

print("Tipos:")
print(df_nullable.dtypes)
print()
print("Datos:")
print(df_nullable)

Tipos:
enteros_con_na      Int64
floats_con_na     Float64
bool_con_na       boolean
texto_con_na       string
dtype: object

Datos:
   enteros_con_na  floats_con_na  bool_con_na texto_con_na
0               1            1.1         True            a
1               2           <NA>         <NA>            b
2            <NA>            3.3        False         <NA>
3               4            4.4         True            d


In [26]:
# Comportamiento con operaciones — NA se propaga o se ignora según la operación
print("Operaciones con nullable dtypes:")
print(f"  sum (enteros):  {df_nullable['enteros_con_na'].sum()}")
print(f"  mean (floats):  {df_nullable['floats_con_na'].mean():.4f}")
print(f"  any (booleans): {df_nullable['bool_con_na'].any()}")
print(f"  all (booleans): {df_nullable['bool_con_na'].all()}")
print()

# Con skipna=False, NA se propaga
print("Con skipna=False (NA se propaga):")
print(f"  sum con NA: {df_nullable['enteros_con_na'].sum(skipna=False)}")

Operaciones con nullable dtypes:
  sum (enteros):  7
  mean (floats):  2.9333
  any (booleans): True
  all (booleans): False

Con skipna=False (NA se propaga):
  sum con NA: <NA>


In [27]:
# Comparación: numpy int64 vs nullable Int64 con NaN
print("numpy int64 con valor faltante — tiene que convertirse a float:")
s_numpy = pd.Series([1, 2, None, 4])  # None fuerza conversión
print(f"  dtype: {s_numpy.dtype}, valores: {s_numpy.values}")
print()

print("nullable Int64 con valor faltante — mantiene tipo entero:")
s_nullable = pd.array([1, 2, None, 4], dtype="Int64")
s_nullable = pd.Series(s_nullable)
print(f"  dtype: {s_nullable.dtype}, valores: {s_nullable.values}")

numpy int64 con valor faltante — tiene que convertirse a float:
  dtype: float64, valores: [ 1.  2. nan  4.]

nullable Int64 con valor faltante — mantiene tipo entero:
  dtype: Int64, valores: <IntegerArray>
[1, 2, <NA>, 4]
Length: 4, dtype: Int64


**Cuándo usar nullable dtypes:**
- Cuando tienes columnas numéricas que pueden tener valores faltantes y el tipo importa (ej: IDs de enteros que no pueden ser float)
- Cuando necesitas lógica booleana de tres valores (True/False/NA)
- En general para datos que vienen de bases de datos SQL donde NULL es común

**Cuándo quedarte con numpy dtypes:**
- Cálculos numéricos intensivos donde `float64` es suficiente
- Interoperabilidad con librerías que no entienden nullable dtypes (sklearn, por ejemplo)

---
## Sección 8: Checklist para migrar de pandas 2.x a pandas 3.0

### Buscar y reemplazar (mecánico, bajo riesgo)
- [ ] `df.applymap(f)` → `df.map(f)`
- [ ] `df1.append(df2)` → `pd.concat([df1, df2])`
- [ ] `df.swapaxes(...)` → refactorizar con transpose o reindex
- [ ] `inplace=True` → `df = df.metodo()`

### Revisar comportamiento (requiere entender el código)
- [ ] **Chained indexing**: `df[cond]["col"] = val` → usar `df.loc[cond, "col"] = val`
- [ ] **groupby sobre categorías**: añadir `observed=True` o `observed=False` explícitamente para dejar la intención clara
- [ ] Código que asume `df.dtypes` tiene `object` para strings → ahora es `string`
- [ ] Código que usa `np.nan` en columnas de texto → ahora es `pd.NA`
- [ ] Tests que verificaban `SettingWithCopyWarning` → ya no se emite

### Verificar antes de deployar
- [ ] `pd.__version__` muestra `3.x.x`
- [ ] Tests de integración pasan (especialmente los que tocan groupby categórico)
- [ ] Resultados de agregaciones sobre categorías son los esperados
- [ ] Código de lectura de CSV produce los tipos esperados

### Herramientas útiles para encontrar problemas
```bash
# Encontrar uso de applymap
grep -r 'applymap' . --include='*.py' --include='*.ipynb'

# Encontrar append en DataFrames (patron heurístico)
grep -r '\.append(' . --include='*.py'

# Encontrar inplace
grep -r 'inplace=True' . --include='*.py'
```

---
## Sección 9: Ejercicio — migrar código legacy

El siguiente código fue escrito para pandas 2.x. Tiene al menos 4 problemas.
Tu tarea: identificarlos y escribir `procesar_datos_v3` que funcione correctamente en pandas 3.0.

In [28]:
# Código legacy — escrito para pandas 2.x
# Identifica los problemas antes de ver la solución

def procesar_datos_legacy(df):
    # Problema 1: chained indexing — con CoW modifica una copia, no df
    df[df["activo"]]["score"] = 100

    # Problema 2: applymap eliminado en pandas 3.0
    df[["precio", "descuento"]] = df[["precio", "descuento"]].applymap(
        lambda x: round(float(x), 2)
    )

    # Problema 3: groupby sin observed — comportamiento cambió en pandas 3.0
    resumen = df.groupby("categoria")["precio"].mean()

    # Problema 4: inplace — deprecado y puede no funcionar
    df.sort_values("precio", inplace=True)

    return df, resumen

print("Código legacy definido (NO ejecutar directamente en pandas 3.0)")

Código legacy definido (NO ejecutar directamente en pandas 3.0)


In [30]:
# DataFrame de prueba para el ejercicio
categoria = pd.Categorical(
    ["A", "B", "A", "C", "B"],
    categories=["A", "B", "C", "D"]  # D no tiene datos
)

df_ejercicio = pd.DataFrame({
    "activo":    [True, False, True, True, False],
    "score":     [0, 0, 0, 0, 0],
    "precio":    [100.123, 200.456, 300.789, 150.111, 250.222],
    "descuento": [0.101, 0.202, 0.303, 0.151, 0.252],
    "categoria": categoria,
})

print("DataFrame de prueba:")
print(df_ejercicio)

DataFrame de prueba:
   activo  score   precio  descuento categoria
0    True      0  100.123      0.101         A
1   False      0  200.456      0.202         B
2    True      0  300.789      0.303         A
3    True      0  150.111      0.151         C
4   False      0  250.222      0.252         B


In [31]:
# Tu versión corregida aquí:
def procesar_datos_v3(df):
    df = df.copy()  # trabajar sobre copia

    # Solución 1: evitar chained indexing usando .loc
    df.loc[df["activo"], "score"] = 100

    # Solución 2: reemplazar applymap (removido en pandas 3) por map columna a columna
    df["precio"] = df["precio"].map(lambda x: round(float(x), 2))
    df["descuento"] = df["descuento"].map(lambda x: round(float(x), 2))

    # Solución 3: especificar observed explícitamente en groupby con categoricals
    resumen = df.groupby("categoria", observed=True)["precio"].mean()

    # Solución 4: eliminar inplace y reasignar
    df = df.sort_values("precio")

    return df, resumen

# Descomenta para probar:
df_resultado, resumen = procesar_datos_v3(df_ejercicio)
print(df_resultado)
print(resumen)

   activo  score  precio  descuento categoria
0    True    100  100.12       0.10         A
3    True    100  150.11       0.15         C
1   False      0  200.46       0.20         B
4   False      0  250.22       0.25         B
2    True    100  300.79       0.30         A
categoria
A    200.455
B    225.340
C    150.110
Name: precio, dtype: float64


In [ ]:
# SOLUCIÓN — no leas esto antes de intentarlo
# (descomenta el bloque para ver la solución)

# def procesar_datos_v3(df):
#     df = df.copy()
#
#     # Fix 1: .loc en lugar de chained indexing
#     df.loc[df["activo"], "score"] = 100
#
#     # Fix 2: df.map en lugar de df.applymap
#     df[["precio", "descuento"]] = df[["precio", "descuento"]].map(
#         lambda x: round(float(x), 2)
#     )
#
#     # Fix 3: observed=True explícito (default en pandas 3.0, pero más claro)
#     resumen = df.groupby("categoria", observed=True)["precio"].mean()
#
#     # Fix 4: reasignar en lugar de inplace
#     df = df.sort_values("precio").reset_index(drop=True)
#
#     return df, resumen
#
# df_resultado, resumen = procesar_datos_v3(df_ejercicio)
# print("DataFrame procesado:")
# print(df_resultado)
# print()
# print("Resumen por categoría (solo las observadas):")
# print(resumen)

print("Solución comentada arriba — descomenta para ver.")